# Point Cloud Aggregation Based on GNSS-INS Data and Moving Object Filtering

This notebook demonstrates:
1. LiDAR point cloud aggregation using ego-motion
2. Moving object removal using temporal voxel consistency
3. Point cloud colorization using camera images

Dataset: nuScenes Mini
Scene: scene-0061


In [1]:
import numpy as np
import open3d as o3d
from nuscenes.nuscenes import NuScenes

from nuscenes_io import load_scene_sweeps, load_scene_camera_frames
from aggregate import aggregate_sweeps_to_global, voxel_downsample
from dynamic_filter import temporal_voxel_static_mask
from colorize import colorize_points_from_cameras


In [5]:
nusc = NuScenes(
    version="v1.0-mini",
    dataroot="/Users/kocsisbarnabas/Documents/Barnabas/Egyetem/3_felev/3D_sensing_and_sensor_fusion/Regular_Assignment_2/v1.0-mini",
    verbose=True,
)

scene_name = "scene-0061"

Loading NuScenes tables for version v1.0-mini...
23 category,
8 attribute,
4 visibility,
911 instance,
12 sensor,
120 calibrated_sensor,
31206 ego_pose,
8 log,
10 scene,
404 sample,
31206 sample_data,
18538 sample_annotation,
4 map,
Done loading in 0.190 seconds.
Reverse indexing ...
Done reverse indexing in 0.0 seconds.


In [4]:
for s in nusc.scene:
    print(s["name"])

scene-0061
scene-0103
scene-0553
scene-0655
scene-0757
scene-0796
scene-0916
scene-1077
scene-1094
scene-1100


## Task 1: Aggregation

In [6]:
sweeps = load_scene_sweeps(nusc, scene_name)
agg = aggregate_sweeps_to_global(sweeps)

print("Aggregated points:", agg.xyz_global.shape[0])

Loading sweeps scene-0061: 100%|██████████| 39/39 [00:00<00:00, 1224.30it/s]

Aggregated points: 1354112


In [7]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(voxel_downsample(agg.xyz_global, 0.2))
o3d.visualization.draw_geometries([pcd])

## Task 2: Moving object filtering

In [8]:
static_mask = temporal_voxel_static_mask(
    agg.xyz_global, agg.sweep_ids, voxel_size=0.2, min_unique_sweeps=4
)

xyz_static = agg.xyz_global[static_mask]
print("Static points:", xyz_static.shape[0])

Static points: 476645


In [9]:
pcd_raw = o3d.geometry.PointCloud()
pcd_raw.points = o3d.utility.Vector3dVector(voxel_downsample(agg.xyz_global, 0.3))
pcd_raw.paint_uniform_color([1, 0, 0])

pcd_static = o3d.geometry.PointCloud()
pcd_static.points = o3d.utility.Vector3dVector(voxel_downsample(xyz_static, 0.3))
pcd_static.paint_uniform_color([0, 1, 0])

o3d.visualization.draw_geometries([pcd_raw, pcd_static])

## Task 3: Colorization

In [10]:
camera_channels = [
    "CAM_FRONT",
    "CAM_FRONT_LEFT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK",
    "CAM_BACK_LEFT",
    "CAM_BACK_RIGHT",
]

cams = load_scene_camera_frames(nusc, scene_name, camera_channels)

xyz_col = voxel_downsample(xyz_static, 0.15)
rgb = colorize_points_from_cameras(xyz_col, cams, nusc)

Loading cameras scene-0061: 100%|██████████| 39/39 [00:00<00:00, 4723.32it/s]


In [11]:
pcd_col = o3d.geometry.PointCloud()
pcd_col.points = o3d.utility.Vector3dVector(xyz_col)
pcd_col.colors = o3d.utility.Vector3dVector(rgb / 255.0)

o3d.visualization.draw_geometries([pcd_col])